In [1]:
import numpy as np

N = 4
n_states = N * N
terminals = {(0, 0), (3, 3)}
gamma = 1.0
actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
V = np.zeros((4, 4))

def step(r, c, dr, dc):
    new_r = r + dr
    new_c = c + dc

    # Pulling values that exceed the boundaries back to the boundary edge, 
    # this handles the actions that go beyond the boundaries.
    new_r = max(0, min(3, new_r))
    new_c = max(0, min(3, new_c))

    # The reward is always -1
    reward = -1

    return ((new_r, new_c), reward)
    
for iteration in range(100):

    for r in range(4):
        for c in range(4):

            #The start and end point detection. These two points act as the base condition for this recursion. 
            #Therefore, even if gamma is 1, there can still be accurate values.
            if (r, c) in terminals:
                    continue

            candidates = []
            for dr, dc in actions:
                (new_r, new_c), reward = step(r, c, dr, dc)
                
                #Bellman equation
                candidate = reward + gamma * (V[new_r, new_c])

                candidates.append(candidate)

            V[r, c] = max(candidates)

print(V)    

[[ 0. -1. -2. -3.]
 [-1. -2. -3. -2.]
 [-2. -3. -2. -1.]
 [-3. -2. -1.  0.]]


In [4]:
arrow = {(-1, 0): "↑", (1, 0): "↓", (0, -1): "←", (0, 1): "→"}

policy = [[""for _ in range(4)]for _ in range(4)]

for r in range(4):
    for c in range(4):

        # The first time I wrote, I forgot to use "continue", 
        #which resulted in not being able to print out "T" and 
        #it was overwritten by the subsequent loop.
        if (r, c) in terminals:
            policy[r][c] = "T"
            continue

        best_value = float('-inf')
        best_action = None

        for (dr, dc) in actions:
            (new_r, new_c), reward = step(r, c, dr, dc)
            candidate = reward + gamma * (V[new_r, new_c])

            if candidate > best_value:
                best_value = candidate
                best_action = (dr, dc)

        #list and dic both use the "[]" to access the value    
        policy[r][c] = arrow[best_action]
        
# "".join can change the connection forms of various string types such as list tuples.
for row in policy:
    print(" ".join(row))
        
        

T ← ← ↓
↑ ↑ ↑ ↓
↑ ↑ ↓ ↓
↑ → → T


In [3]:
print(step(2, 2, -1, 0))   
print(step(0, 0, -1, 0))  
print(step(3, 3, 0, 1))  

((1, 2), -1)
((0, 0), -1)
((3, 3), -1)


In [1]:
import gymnasium as gym
print(gym.__version__)

1.3.0


In [5]:
env = gym.make("CartPole-v1")

#print("action_space:", env.action_space)
#print("observation_space:", env.observation_space)

observation, info = env.reset(seed = 42)
print("state is", observation)
print("style is", type(observation))
print("shape is", observation.shape)
print("info is", info)

action = 1
next_observation, reward, terminated, truncated, info = env.step(action)

print("action", action)
print("new observation is", next_observation)
print("reward is", reward)
print("terminated", terminated)
print("truncated", truncated)

state is [ 0.0273956  -0.00611216  0.03585979  0.0197368 ]
style is <class 'numpy.ndarray'>
shape is (4,)
info is {}
action 1
new observation is [ 0.02727336  0.18847767  0.03625453 -0.26141977]
reward is 1.0
terminated False
truncated False


In [10]:
observation, info = env.reset(seed = 42)

total_reward = 0
steps = 0

while True:
    action = env.action_space.sample()

    next_observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1

    if terminated or truncated:
        break

    observation = next_observation

print(f"This round lasted for {steps}, total_reward is {total_reward}")
    

This round lasted for 12, total_reward is 12.0


In [18]:
import numpy as np

episode_reward = []

for episode in range(20):

    # The first time I wrote it without including this, the reward remained at 0
    observation, info = env.reset()
    total_reward = 0
    steps = 0

    while True:
        
        action = env.action_space.sample()
        next_observation, reward, terminated, truncated, info = env.step(action)
        
        total_reward += reward
        steps += 1
        
        if terminated or truncated:
            break
            
        observation = next_observation
    # episode_reward = episode_reward.append(total_reward), This way of writing is incorrect.
    episode_reward.append(total_reward)

episode_reward = np.array(episode_reward)
print(f"20 episodes average reward is {episode_reward.mean():.1f}")
print(f"the standard of reward is {episode_reward.std():.1f}")
print(f"the max is {episode_reward.max()}")
print(f"the min is {episode_reward.min()}")
    

        

20 episodes average reward is 22.4
the standard of reward is 12.1
the max is 54.0
the min is 10.0


In [1]:
import numpy as np

gamma = 1
alpha = 1

v = np.zeros(3)

def step(i):
    if i == 2:
        reward = 10
        next_value = 0
        return reward, next_value

    else:
        reward = 0
        next_value = v[i + 1]
        return reward, next_value

print(v)


[0. 0. 0.]


In [2]:
for episode in range(3):

    for i in range(3):
        reward, next_value = step(i)
        td_target = reward + gamma * next_value

        v[i] = v[i] + alpha * (td_target - v[i])

    print(f"after{episode + 1}, v = {v}")

after1, v = [ 0.  0. 10.]
after2, v = [ 0. 10. 10.]
after3, v = [10. 10. 10.]


In [1]:
import numpy as np
import gymnasium as gym

env = gym.make("CartPole-v1")
state, info = env.reset()

N_BINS = (6, 6, 12, 12)

STATE_BOUNDS = [
    (-4.8, 4.8),
    (-3.0, 3.0),
    (-0.418, 0.418),
    (-3.5, 3.5),
]

def discretize(state):

    indices = []
    for i in range(4):
        low, high = STATE_BOUNDS[i]
        bins = np.linspace(low, high, N_BINS[i] + 1)[1: -1]
        idx = np.digitize(state[i], bins)
        
        indices.append(idx)
    return tuple(indices)

    
def choose_action(state, epsilon):

    if np.random.random() < epsilon:
        action = env.action_space.sample()

    else:
        s = discretize(state)
        action = np.argmax(Q[s])
    return action


n_actions = env.action_space.n
q_table_shape =N_BINS + (n_actions,)
Q = np.zeros(q_table_shape)

#print("Continuous state", state)
#print("after discretization", discretize(state))
#print("Q-table shape:", Q.shape)
print("Pure exploitation:", choose_action(state, 0.0))
print("Pure exploration:", [choose_action(state, 1.0) for _ in range(10)])


Pure exploitation: 0
Pure exploration: [np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0)]


In [2]:
alpha = 0.1
gamma = 0.99

def update(state, next_state, action, reward, done):
    s = discretize(state)
    s_next = discretize(next_state)

    if done:
        td_target = reward

    else:
        best_next = np.max(Q[s_next])
        td_target = reward + gamma * best_next

    td_error = td_target - Q[s][action]
    Q[s][action] = Q[s][action] + alpha * td_error

In [5]:
n_episode = 2000
epsilon = 1.0
epsilon_min = 0.05
decay = 0.99

for episode in range(n_episode):
    state, info = env.reset()
    total_reward = 0
    done = False
    episode_reward = []

    while not done:

        action = choose_action(state, epsilon)
        (next_state, reward, terminated, truncated, info) = env.step(action)

        update(state, next_state, action, reward, done)

        state = next_state
        total_reward +=reward
        done = terminated or truncated
        
    epsilon = max(epsilon_min, epsilon * decay)
    episode_reward.append(total_reward)


    if episode % 100 == 0:   
        print(f"Episode {episode}, epsilon {epsilon:.3f}, reward {total_reward}")

Episode 0, epsilon 0.990, reward 12.0
Episode 100, epsilon 0.362, reward 19.0
Episode 200, epsilon 0.133, reward 19.0
Episode 300, epsilon 0.050, reward 23.0
Episode 400, epsilon 0.050, reward 24.0
Episode 500, epsilon 0.050, reward 17.0
Episode 600, epsilon 0.050, reward 18.0
Episode 700, epsilon 0.050, reward 24.0
Episode 800, epsilon 0.050, reward 286.0
Episode 900, epsilon 0.050, reward 185.0
Episode 1000, epsilon 0.050, reward 32.0
Episode 1100, epsilon 0.050, reward 24.0
Episode 1200, epsilon 0.050, reward 31.0
Episode 1300, epsilon 0.050, reward 85.0
Episode 1400, epsilon 0.050, reward 186.0
Episode 1500, epsilon 0.050, reward 26.0
Episode 1600, epsilon 0.050, reward 119.0
Episode 1700, epsilon 0.050, reward 21.0
Episode 1800, epsilon 0.050, reward 82.0
Episode 1900, epsilon 0.050, reward 154.0
